# Phase 2: Dataset Understanding

**Project:** Automated Data Quality & ETL Framework  
**Dataset:** Online Retail.xlsx  

This notebook profiles the raw dataset before cleaning or loading it into MySQL. The goal is to understand structure, data types, missing values, duplicates, invalid records, and initial business rules.


## Step 1: Import Required Libraries

`pandas` is used for data handling.  
`numpy` is used for numerical operations.  
`pathlib` is used to manage file paths cleanly.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path


## Step 2: Define Project Paths

Update only `project_folder` if your project location changes. The remaining paths are built automatically.


In [ ]:
# Main project folder
project_folder = Path(r"C:\Users\chatu\OneDrive\Desktop\Automated-Data-Quality-ETL-Framewor")

# Input file path
raw_file_path = project_folder / "data" / "raw" / "Online Retail.xlsx"

# Output folders
reports_folder = project_folder / "reports"
reports_folder.mkdir(exist_ok=True)

# Output files
profiling_report_path = reports_folder / "Data_Profiling_Report.xlsx"
sample_preview_path = reports_folder / "sample_preview.csv"

print("Raw file path:", raw_file_path)
print("Reports folder:", reports_folder)


## Step 3: Load the Dataset

The dataset is loaded from the raw folder. We do not modify the raw file.


In [ ]:
# Read the Excel dataset
# Note: This may take a few seconds because the file has more than 500,000 rows.
df = pd.read_excel(raw_file_path)

# Display the first five rows
df.head()


## Step 4: Check Dataset Size

This shows the number of rows and columns in the dataset.


In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])


## Step 5: Check Column Names and Data Types

This helps identify whether columns are loaded correctly, especially dates and numeric fields.


In [ ]:
# Column names
list(df.columns)


In [ ]:
# Data types of each column
df.dtypes


In [ ]:
# Dataset information: data types, non-null counts, and memory usage
df.info()


## Step 6: Missing Value Analysis

This identifies columns with null values and calculates the missing percentage.


In [ ]:
missing_values = pd.DataFrame({
    "Column": df.columns,
    "Missing_Count": df.isnull().sum().values,
    "Missing_Percentage": ((df.isnull().sum() / len(df)) * 100).round(2).values
})

missing_values = missing_values.sort_values(by="Missing_Count", ascending=False)
missing_values


## Step 7: Duplicate Row Analysis

This checks whether the dataset contains exact duplicate rows.


In [ ]:
duplicate_count = df.duplicated().sum()
duplicate_percentage = round((duplicate_count / len(df)) * 100, 2)

print("Duplicate Rows:", duplicate_count)
print("Duplicate Percentage:", duplicate_percentage, "%")


## Step 8: Unique Value Analysis

This shows the number of distinct values in each column. It helps identify high-cardinality columns like invoice number, stock code, and customer ID.


In [ ]:
unique_values = pd.DataFrame({
    "Column": df.columns,
    "Unique_Count": df.nunique(dropna=True).values
})

unique_values = unique_values.sort_values(by="Unique_Count", ascending=False)
unique_values


## Step 9: Numerical Summary

This summarizes numeric columns using count, mean, standard deviation, minimum, maximum, and quartiles.


In [ ]:
df.describe()


## Step 10: Categorical Summary

This summarizes text columns using count, unique values, most frequent value, and frequency.


In [ ]:
df.describe(include="object")


## Step 11: Business Rule Checks

These checks identify records that may require cleaning or business interpretation in Phase 3.


In [ ]:
# Negative quantity may represent product returns or cancellations.
negative_quantity = df[df["Quantity"] < 0]

print("Negative Quantity Records:", negative_quantity.shape[0])
negative_quantity.head()


In [ ]:
# Unit price should usually be greater than 0 for valid sales transactions.
invalid_price = df[df["UnitPrice"] <= 0]

print("Zero or Negative UnitPrice Records:", invalid_price.shape[0])
invalid_price.head()


In [ ]:
# Missing CustomerID means the transaction cannot be linked to a known customer.
missing_customer_id = df[df["CustomerID"].isnull()]

print("Missing CustomerID Records:", missing_customer_id.shape[0])
missing_customer_id.head()


In [ ]:
# Check invoice date range.
# This helps identify whether the data covers the expected time period.
print("Minimum Invoice Date:", df["InvoiceDate"].min())
print("Maximum Invoice Date:", df["InvoiceDate"].max())


In [ ]:
# Check for future invoice dates compared to today's system date.
future_dates = df[df["InvoiceDate"] > pd.Timestamp.today()]

print("Future Invoice Date Records:", future_dates.shape[0])


## Step 12: Distribution Checks

These checks help understand country distribution and product frequency.


In [ ]:
# Number of transactions by country
df["Country"].value_counts().head(20)


In [ ]:
# Most frequent product descriptions
df["Description"].value_counts().head(20)


## Step 13: Create Revenue Column for Profiling

Revenue is calculated as `Quantity × UnitPrice`. This is only for profiling at this stage.


In [ ]:
df["Revenue"] = df["Quantity"] * df["UnitPrice"]

df[["Quantity", "UnitPrice", "Revenue"]].head()


In [ ]:
# Negative revenue usually comes from negative quantity records.
negative_revenue = df[df["Revenue"] < 0]

print("Negative Revenue Records:", negative_revenue.shape[0])
negative_revenue.head()


## Step 14: Create Data Dictionary

The data dictionary documents each column, its data type, missing values, and unique values.


In [ ]:
data_dictionary = pd.DataFrame({
    "Column_Name": df.columns,
    "Data_Type": df.dtypes.astype(str).values,
    "Non_Null_Count": df.notnull().sum().values,
    "Missing_Count": df.isnull().sum().values,
    "Missing_Percentage": ((df.isnull().sum() / len(df)) * 100).round(2).values,
    "Unique_Count": df.nunique(dropna=True).values
})

data_dictionary


## Step 15: Create Data Quality Issue Summary

This summary gives a high-level issue count and severity classification.


In [ ]:
dq_summary = pd.DataFrame({
    "Issue": [
        "Duplicate Rows",
        "Missing CustomerID",
        "Missing Description",
        "Negative Quantity",
        "Zero or Negative UnitPrice",
        "Negative Revenue",
        "Future Invoice Dates"
    ],
    "Count": [
        df.duplicated().sum(),
        df["CustomerID"].isnull().sum(),
        df["Description"].isnull().sum(),
        (df["Quantity"] < 0).sum(),
        (df["UnitPrice"] <= 0).sum(),
        (df["Revenue"] < 0).sum(),
        (df["InvoiceDate"] > pd.Timestamp.today()).sum()
    ],
    "Severity": [
        "Medium",
        "High",
        "Medium",
        "High",
        "High",
        "High",
        "Critical"
    ]
})

dq_summary


## Step 16: Export Data Profiling Report

This exports all key Phase 2 outputs into one Excel workbook with multiple sheets.


In [ ]:
with pd.ExcelWriter(profiling_report_path, engine="openpyxl") as writer:
    data_dictionary.to_excel(writer, sheet_name="Data Dictionary", index=False)
    missing_values.to_excel(writer, sheet_name="Missing Values", index=False)
    unique_values.to_excel(writer, sheet_name="Unique Values", index=False)
    dq_summary.to_excel(writer, sheet_name="DQ Issue Summary", index=False)
    df.describe().to_excel(writer, sheet_name="Numerical Summary")
    df.describe(include="object").to_excel(writer, sheet_name="Categorical Summary")

print("Data Profiling Report created successfully:", profiling_report_path)


## Step 17: Save Sample Preview File

This creates a small CSV sample of the first 1,000 rows for quick review or sharing.


In [ ]:
df.head(1000).to_csv(sample_preview_path, index=False)

print("Sample preview file created successfully:", sample_preview_path)


# Phase 2 Completion Notes

Phase 2 is complete after this notebook creates:

1. `Data_Profiling_Report.xlsx`
2. `sample_preview.csv`

These outputs will guide Phase 3: Data Cleaning and Validation.
